# 📈 Notebook 2: SARIMAX Baseline Model
**Project:** A Hybrid Deep Learning Approach for Modelling Global CO₂ Emissions  
**Author:** Hafiza Alishba Naaz | NUST Islamabad 2026

> ⚠️ **Run Notebook 1 first** to generate `data/top_10_emitters.csv`

## 2.1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error

print('✅ Libraries imported successfully!')

## 2.2 Load Data & Configuration

In [ ]:
df_energy = pd.read_csv('../data/global-data-on-sustainable-energy.csv')

y_col = 'Value_co2_emissions_kt_by_country'
exog_cols = [
    'gdp_per_capita',
    'gdp_growth',
    'Primary energy consumption per capita (kWh/person)',
    'Renewable energy share in the total final energy consumption (%)'
]

TRAIN_END = 2014
FORECAST_STEPS = 5

top10_countries = (
    df_energy.groupby('Entity')[y_col]
    .mean().dropna()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

print(f'Top 10 countries: {top10_countries}')
print(f'Train period: 2000–{TRAIN_END}')
print(f'Test period: {TRAIN_END+1}–{TRAIN_END+FORECAST_STEPS}')

## 2.3 Stationarity Testing (ADF Test)

In [ ]:
def adf_test(series, label='Series'):
    result = adfuller(series.dropna())
    print(f'--- ADF Test: {label} ---')
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'p-value       : {result[1]:.4f}')
    stationary = result[1] <= 0.05
    print(f'Result: {"STATIONARY ✅" if stationary else "NON-STATIONARY ❌"}\n')
    return result[1]

def get_d_order(series):
    if adfuller(series.dropna())[1] <= 0.05: return 0
    if adfuller(series.diff().dropna())[1] <= 0.05: return 1
    return 2

print('Stationarity Results — All 10 Countries:')
print('='*60)
for country in top10_countries:
    sub = df_energy[df_energy['Entity'] == country][['Year', y_col]].dropna().sort_values('Year')
    series = sub.set_index('Year')[y_col]
    d = get_d_order(series)
    print(f'{country:20s} | Best d = {d}')

## 2.4 ACF & PACF — China Example

In [ ]:
china_sub = df_energy[df_energy['Entity']=='China'][['Year', y_col]].dropna().sort_values('Year')
china_ser = china_sub.set_index('Year')[y_col]
china_d1  = china_ser.diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_acf(china_ser,  lags=10, ax=axes[0,0], title='ACF — Original')
plot_pacf(china_ser, lags=8,  ax=axes[0,1], title='PACF — Original')
plot_acf(china_d1,   lags=10, ax=axes[1,0], title='ACF — First Difference')
plot_pacf(china_d1,  lags=8,  ax=axes[1,1], title='PACF — First Difference')
for ax in axes.flatten(): ax.grid(alpha=0.3)
plt.suptitle('ACF & PACF — China CO₂ Emissions', fontsize=12)
plt.tight_layout()
plt.savefig('../results/acf_pacf_china.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Auto ARIMA Order Selection

In [ ]:
best_orders = {}
print('Auto ARIMA Order Selection (AIC criterion)')
print('='*55)
for country in top10_countries:
    sub = (df_energy[df_energy['Entity'] == country]
           [['Year', y_col] + exog_cols]
           .dropna().sort_values('Year').set_index('Year'))
    train = sub[sub.index <= TRAIN_END]
    d = get_d_order(train[y_col])
    aa = auto_arima(train[y_col], X=train[exog_cols],
                    start_p=0, max_p=3, start_q=0, max_q=3,
                    d=d, information_criterion='aic',
                    stepwise=True, suppress_warnings=True, error_action='ignore')
    best_orders[country] = aa.order
    print(f'{country:20s} | Best order = {aa.order} | AIC = {aa.aic():.1f}')

## 2.6 Fit SARIMAX — All 10 Countries

In [ ]:
all_results   = []
all_forecasts = {}
all_models    = {}

def extrapolate_exog(exog_df, steps):
    future = {}
    x = np.arange(len(exog_df))
    for col in exog_df.columns:
        coeffs = np.polyfit(x, exog_df[col].values, 1)
        future_x = np.arange(len(exog_df), len(exog_df) + steps)
        future[col] = np.polyval(coeffs, future_x)
    return pd.DataFrame(future, index=range(exog_df.index[-1]+1, exog_df.index[-1]+1+steps))

for country in top10_countries:
    sub   = (df_energy[df_energy['Entity'] == country]
             [['Year', y_col] + exog_cols]
             .dropna().sort_values('Year').set_index('Year'))
    train = sub[sub.index <= TRAIN_END]
    test  = sub[sub.index >  TRAIN_END]
    order = best_orders[country]
    try:
        model = SARIMAX(train[y_col], exog=train[exog_cols],
                        order=order, seasonal_order=(0,0,0,0),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False, maxiter=500)
        preds = fit.predict(start=len(train), end=len(train)+len(test)-1, exog=test[exog_cols])
        preds = pd.Series(preds.values, index=test.index)
        rmse  = np.sqrt(mean_squared_error(test[y_col], preds))
        mae   = mean_absolute_error(test[y_col], preds)
        mape  = np.mean(np.abs((test[y_col].values - preds.values) / test[y_col].values)) * 100
        r2    = 1 - np.sum((test[y_col].values - preds.values)**2) / np.sum((test[y_col].values - test[y_col].mean())**2)
        lb    = acorr_ljungbox(fit.resid.dropna(), lags=[5], return_df=True)
        lb_p  = lb['lb_pvalue'].values[0]
        all_results.append({'Country': country, 'RMSE': round(rmse,0), 'MAE': round(mae,0),
                            'MAPE_%': round(mape,2), 'R2': round(r2,4),
                            'LjungBox_p': round(lb_p,4), 'WhiteNoise': 'Yes' if lb_p>0.05 else 'No'})
        all_forecasts[country] = {'train': train, 'test': test, 'preds': preds}
        all_models[country]    = fit
        print(f'{country:20s} | MAPE={mape:5.2f}% | R²={r2:.4f} | LB_p={lb_p:.3f}')
    except Exception as e:
        print(f'{country:20s} | ERROR: {e}')

results_df = pd.DataFrame(all_results)
print('\nMean MAPE:', results_df['MAPE_%'].mean().round(2))
print('Mean R²  :', results_df['R2'].mean().round(4))
results_df.to_csv('../results/sarimax_results.csv', index=False)
print('\n✅ Saved: results/sarimax_results.csv')

## 2.7 SARIMAX Forecast Plots — All Countries

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
colors_plot = ['#378ADD','#1D9E75','#D85A30','#7F77DD','#BA7517',
               '#533AB7','#0F6E56','#993C1D','#639922','#3C3489']

for ax, country, color in zip(axes.flatten(), top10_countries, colors_plot):
    fc = all_forecasts[country]
    r  = results_df[results_df['Country'] == country].iloc[0]
    ax.plot(fc['train'].index, fc['train'][y_col], color='#AAAAAA', lw=1.5, marker='o', markersize=3, label='Train')
    ax.plot(fc['test'].index,  fc['test'][y_col],  color=color,     lw=2,   marker='o', markersize=5, label='Actual')
    ax.plot(fc['preds'].index, fc['preds'].values, color='#D85A30', lw=2,   marker='s', markersize=5, linestyle='--', label='SARIMAX')
    ax.axvline(TRAIN_END, color='gray', linestyle=':', lw=1)
    ax.set_title(f"{country}\nMAPE={r['MAPE_%']:.1f}% | R²={r['R2']:.3f}", fontsize=8)
    ax.set_xlabel('Year', fontsize=7); ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3); ax.legend(fontsize=6)

plt.suptitle('SARIMAX Forecasts — Top 10 CO₂ Emitters (2015–2019)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/sarimax_all_countries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pickle
with open('../data/sarimax_objects.pkl', 'wb') as f:
    pickle.dump({'all_models': all_models, 'all_forecasts': all_forecasts,
                 'top10_countries': top10_countries}, f)